# RK1828 RAFT-Stereo 双目视差 Demo


[RAFT-Stereo](https://github.com/princeton-vl/RAFT-Stereo)（普林斯顿视觉与学习实验室）
从一对左右目图片估计**视差图**（disparity）：越亮表示越近。
这里用的是 realtime 版本，输入 160×240，int8 量化，RK1828 上约 90ms/对。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 运行并查看视差图 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("raft-stereo-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/raft_stereo/demo_x86_64.tar.gz"),
    ("raft-stereo-rknn3-rk1828", "model/raftstereo-realtime-160x240.rknn", "demos/raft_stereo/model/raftstereo-realtime-160x240.rknn"),
    ("raft-stereo-rknn3-rk1828", "model/raftstereo-realtime-160x240.weight", "demos/raft_stereo/model/raftstereo-realtime-160x240.weight"),
    ("raft-stereo-rknn3-rk1828", "model/xt_left.jpg", "demos/raft_stereo/model/xt_left.jpg"),
    ("raft-stereo-rknn3-rk1828", "model/xt_right.jpg", "demos/raft_stereo/model/xt_right.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/raft_stereo/demo_x86_64.tar.gz", "demos/raft_stereo")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 运行 demo

输入：`model/xt_left.jpg` / `model/xt_right.jpg`（一对示例双目图）
输出：`out.png` 视差热力图

In [ ]:
show_images(["demos/raft_stereo/model/xt_left.jpg",
             "demos/raft_stereo/model/xt_right.jpg"])

In [ ]:
DEMO = "demos/raft_stereo"
before = snapshot(DEMO)
run_demo(["./rknn_raft_stereo_demo",
          "model/raftstereo-realtime-160x240.rknn",
          "model/raftstereo-realtime-160x240.weight",
          "model/xt_left.jpg", "model/xt_right.jpg", "0x1"], DEMO)
show_new_images(DEMO, before)

## 小结

- 视差 = 同一物体在左右图中的水平位移，视差越大距离越近，是双目深度估计的核心。
- 可换成自己的双目照片（保持 160×240 或由 demo 内部缩放）。